In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Fasttext

In [ ]:
f = open("/content/drive/MyDrive/파일위치/book_law_data_kkm.txt", 'r')
sList = []
lines = f.readlines()
for line in lines:
    line = line.strip()  # 줄 끝의 줄 바꿈 문자를 제거한다.
    sList.append(line)
f.close()
print(sList[:5])

['부정 청탁   문제 유출   입학 취소', '임금 표   임금 등급   노동 기준 을 사업장 에 미 공표', '가 아 등기 의 개념', '북한 학자 들 이 쓰   저서 또는 논문 의 입수 가 어렵   현 단계 에서 그 들 의 국제법 에 대하   태도 를 개관 하  다는 것 은 불가능 하   일 이 다', '그러나 몇 편의 논문 및 사전 류     등 을 보 건대 그 들 의 서술 방식 이 특이 함 을 느끼 게 되  다']


In [ ]:
result = [s.split(' ') for s in sList]
print(result[:3])

[['부정', '청탁', '', '', '문제', '유출', '', '', '입학', '취소'], ['임금', '표', '', '', '임금', '등급', '', '', '노동', '기준', '을', '사업장', '에', '미', '공표'], ['가', '아', '등기', '의', '개념']]


In [ ]:
for l in result:
    while True:
        try:
            l.remove('')
        except:
            break
print(result[:3])

[['부정', '청탁', '문제', '유출', '입학', '취소'], ['임금', '표', '임금', '등급', '노동', '기준', '을', '사업장', '에', '미', '공표'], ['가', '아', '등기', '의', '개념']]


In [ ]:
from gensim.models import FastText
#각 형태소 분석기 별로 수행 후 저장
model = FastText(result, vector_size=100, window=5, min_count=1, workers=4, sg=0)
model.save('/content/drive/MyDrive/파일위치/book_law_kkm_fst.model')

In [ ]:
#임베딩하는 코드
##시용자 입력이 들어오면 (들어올 때마다) 사용자 입력과 전체 질문리스트(혹은 해당 카테고리의 질문 리스트)를 임베딩해줌
from tqdm import tqdm
def get_document_vectors(document_list):
    document_embedding_list = []
    for line in tqdm(document_list):
        doc2vec = None
        count = 0
        for word in line.split():
            #print(word)
            try:
                w = model.wv[word]
                count += 1

                #해당 문서에 있는 모든 단어들의 벡터 값을 더한다.
                if doc2vec is None :
                    doc2vec = w
                else :
                    doc2vec = doc2vec + w
            except:
                pass

        if doc2vec is not None:
            # 단어 벡터를 모두 더한 벡터의 값을 문서 길이로 나눠준다.
            doc2vec = doc2vec/count
            document_embedding_list.append(doc2vec)
        else:
            document_embedding_list.append(None)

    # 각 문서에 대한 문서 벡터 리스트를 리턴
    return document_embedding_list


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/파일위치/pan_qna_trans_kkm_w2v.csv')
df.tail()

In [ ]:
emd = get_document_vectors(df['question'])
emd2 = get_document_vectors(df['trans_question'])

In [ ]:
df['embedding']= emd
df['trans_q_embedding'] = emd2
df.to_csv('/content/drive/MyDrive/파일위치/pan_qna_kkm_trans_fst.csv')

#sbert

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/xlm-r-100langs-bert-base-nli-stsb-mean-tokens')

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/파일위치/pan_qna_trans.csv')

In [ ]:
from tqdm import tqdm
emd = []
emd2 = []
for i in tqdm(range(len(df))):
    emd.append(model.encode(df['question'][i]))
    emd2.append(model.encode(df['trans_question'][i]))
df['embedding'] = emd
df['trans_q_embedding'] = emd2
df.to_csv('/content/drive/MyDrive/파일위치/pan_qna_trans_sbert.csv')